Calculate ii and vv distance of z-threshold patterns

In [1]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    auc,
)
from toolbox import (
    Masking,
    compute_ROC_patient_split,
    find_all_indices,
    hot_coding,
)
from toolbox.loading import load_data, load_data_no_norm
from toolbox.visualization import cmap_gradient

In [4]:
# load masking
IIVV = np.loadtxt("source/IIVV_patterns.txt", dtype=int)
mask = Masking(IIVV)
# set prints off
mask.prints = False
# load in average impedances 
avg_Z_fp = np.load('Avg_Zs.npy')

IIVV.shape

(7728, 4)

In [5]:
# functions from masking 
def euclidean_distance(el1, el2):
    """
    Calculate Euclidean distance for single or arrays of electrodes.

    Parameters
    ----------
    el1 : int or array-like of ints
        Electrode indices (can be scalar or numpy array)
    el2 : int or array-like of ints
        Electrode indices (same shape as el1)

    Returns
    -------
    np.ndarray or float
        Distances between electrode pairs. If inputs are arrays, returns array
        of distances; else returns single float distance.
    """
    def index_to_coordinates(index):
        spacing = 1.0  # 1 mm spacing
        if np.isscalar(index):
            # Single electrode index
            if 1 <= index <= 25:
                row = (index - 1) // 5 + 1
                col = (index - 1) % 5 + 1
                x = (col - 3) * spacing
                y = (3 - row) * spacing
                return np.array([x, y])
            elif 26 <= index <= 33:
                current_coords = {
                    26: np.array([-3,  1]),
                    27: np.array([-3, -1]),
                    28: np.array([-1,  3]),
                    29: np.array([ 1,  3]),
                    30: np.array([ 3,  1]),
                    31: np.array([ 3, -1]),
                    32: np.array([ 1, -3]),
                    33: np.array([-1, -3]),
                }
                return current_coords.get(index, np.array([np.nan, np.nan]))
            else:
                return np.array([np.nan, np.nan])
        else:
            # Vectorized coordinates
            coords = np.empty((len(index), 2), dtype=float)
            for i, idx in enumerate(index):
                coords[i] = index_to_coordinates(idx)
            return coords

    # Convert to numpy arrays
    el1 = np.atleast_1d(el1)
    el2 = np.atleast_1d(el2)

    coords1 = index_to_coordinates(el1)
    coords2 = index_to_coordinates(el2)

    # Compute Euclidean distances element-wise
    diffs = coords1 - coords2
    dists = np.linalg.norm(diffs, axis=1)

    # If input is scalar, return scalar
    if dists.size == 1:
        return dists.item()
    return dists

In [8]:
import numpy as np

class_task = {'healthy_vs_cancer', 'cancer_vs_hbd', 'MC'}

for task in class_task: 
    
    if task == "healthy_vs_cancer":
        print("Task 1 Z-threshold: 386")
        z_thresh = 386 
    elif task == 'cancer_vs_hbd':
        print("Task 2 Z-threshold: 423")
        z_thresh = 423 
    elif task == 'MC':
        print("Task 3 Z-threshold: 592")
        z_thresh = 592

    Msel = f"z_threshold: {z_thresh}"
    
    # Select patterns below threshold
    below_threshold = avg_Z_fp[0, :] <= z_thresh

    # Filter IIVV rows (patterns)
    selected_pairs = IIVV[below_threshold, :]  # shape (n_selected, 4)

    # Extract electrode indices for II and VV pairs
    i0 = selected_pairs[:, 0]
    i1 = selected_pairs[:, 1]

    v1 = selected_pairs[:, 2]
    v2 = selected_pairs[:, 3]

    # Compute distances
    ii_distances = euclidean_distance(i0, i1)
    vv_distances = euclidean_distance(v1, v2)

    # Calculate averages
    average_ii_distance = np.mean(ii_distances)
    average_vv_distance = np.mean(vv_distances)

    print(f"{task} - Average II distance (z <= {z_thresh}): {average_ii_distance:.3f}")
    print(f"{task} - Average VV distance (z <= {z_thresh}): {average_vv_distance:.3f}")



Task 2 Z-threshold: 423
cancer_vs_hbd - Average II distance (z <= 423): 4.116
cancer_vs_hbd - Average VV distance (z <= 423): 2.186
Task 3 Z-threshold: 592
MC - Average II distance (z <= 592): 4.194
MC - Average VV distance (z <= 592): 2.255
Task 1 Z-threshold: 386
healthy_vs_cancer - Average II distance (z <= 386): 4.093
healthy_vs_cancer - Average VV distance (z <= 386): 2.164
